# VeloceAI — Colab Free (one-click launch)
1. اختَر GPU من Runtime → Change runtime type إذا كان متاحًا (اختياري؛ CPU أبطأ).
2. شغّل خلية التشغيل الوحيدة بالأسفل وانتظر ظهور رابط VeloceAI.
3. افتح الرابط واترك هذه الجلسة تعمل أثناء التحسين. الرابط مؤقت ويتوقف عند انتهاء جلسة Colab.

لا تحتاج Hugging Face أو اشتراكًا مدفوعًا. النموذج يُحمّل تلقائيًا عند أول عملية تحسين.

In [ ]:
import os, sys, time, re, pathlib, subprocess, urllib.request, json

def run(command):
    print('Running:', ' '.join(command[:5]), '...')
    subprocess.run(command, check=True)

# Install only the app dependencies; keep Colab's preinstalled PyTorch build.
run(['apt-get','-qq','update'])
run(['apt-get','-qq','install','-y','ffmpeg'])
run([sys.executable,'-m','pip','install','-q','fastapi','uvicorn[standard]','python-multipart','pillow','opencv-python-headless','spandrel'])

base='https://raw.githubusercontent.com/rezeyuri-dot/veloceai/main/'
def fetch_latest(name, destination):
    request=urllib.request.Request(
        base+name+'?refresh='+str(time.time_ns()),
        headers={'Cache-Control':'no-cache','Pragma':'no-cache'})
    with urllib.request.urlopen(request,timeout=45) as response:
        pathlib.Path(destination).write_bytes(response.read())
fetch_latest('colab_server.py','/content/colab_server.py')
fetch_latest('index.html','/content/index.html')

# Stop old processes if this launch cell is run again.
for name in ('tunnel','server'):
    old=globals().get(name)
    try:
        if old and old.poll() is None:
            old.terminate(); old.wait(timeout=8)
    except Exception:
        pass

server_log=open('/content/veloceai-server.log','w')
server=subprocess.Popen([sys.executable,'/content/colab_server.py'],stdout=server_log,stderr=subprocess.STDOUT)
for _ in range(45):
    try:
        health=json.loads(urllib.request.urlopen('http://127.0.0.1:8000/health',timeout=3).read())
        if health.get('ok'): break
    except Exception: pass
    time.sleep(2)
else:
    print(pathlib.Path('/content/veloceai-server.log').read_text(errors='ignore')[-5000:])
    raise RuntimeError('VeloceAI server did not start. See the log above.')

cloudflared='/usr/local/bin/cloudflared'
if not pathlib.Path(cloudflared).exists():
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',cloudflared)
    os.chmod(cloudflared,0o755)
tunnel_log=open('/content/veloceai-tunnel.log','w')
tunnel=subprocess.Popen([cloudflared,'tunnel','--protocol','http2','--url','http://127.0.0.1:8000','--no-autoupdate'],stdout=tunnel_log,stderr=subprocess.STDOUT)
public_url=None
for _ in range(60):
    time.sleep(2)
    log=pathlib.Path('/content/veloceai-tunnel.log').read_text(errors='ignore')
    match=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',log)
    if match:
        public_url=match.group(0); break
if not public_url:
    print(pathlib.Path('/content/veloceai-tunnel.log').read_text(errors='ignore')[-5000:])
    raise RuntimeError('Could not obtain a temporary website URL.')

print('\n'+'='*60)
print('Local server health:',urllib.request.urlopen('http://127.0.0.1:8000/health',timeout=10).read().decode())
print('Loading AI model during startup (first launch may take a little longer)...')
try:
    warmup_request=urllib.request.Request('http://127.0.0.1:8000/warmup?profile=anime',data=b'',method='POST')
    warmup_result=json.loads(urllib.request.urlopen(warmup_request,timeout=600).read())
    print('Anime model ready:',warmup_result)
    standard_request=urllib.request.Request('http://127.0.0.1:8000/warmup?profile=general',data=b'',method='POST')
    standard_result=json.loads(urllib.request.urlopen(standard_request,timeout=600).read())
    print('General/Real model ready:',standard_result)
except Exception as e:
    raise RuntimeError('AI model warm-up failed: '+repr(e))

# Real API smoke tests on tiny synthetic media, including settings, video FPS and audio.
import io, json, requests, numpy as np
from PIL import Image, ImageDraw
from IPython.display import display, Video
test_dir=pathlib.Path('/content/veloceai-self-test')
test_dir.mkdir(exist_ok=True)
yy,xx=np.mgrid[0:64,0:64]
pattern=np.zeros((64,64,3),dtype=np.uint8)
pattern[:,:,0]=np.clip(xx*4,0,255)
pattern[:,:,1]=np.clip(yy*4,0,255)
pattern[:,:,2]=((xx//8%2)^(yy//8%2))*170+40
sample=Image.fromarray(pattern,'RGB')
draw=ImageDraw.Draw(sample)
draw.rectangle((12,12,51,51),outline=(255,255,255),width=2)
draw.ellipse((22,22,41,41),fill=(245,80,120))
image_buffer=io.BytesIO()
sample.save(image_buffer,format='PNG')
image_bytes=image_buffer.getvalue()

image_cases=[
    ('anime-2x-default',{'scale':2,'denoise':0,'sharpness':1.0,'profile':'anime','line_detail':20,'anti_alias':0}),
    ('anime-4x-max-controls',{'scale':4,'denoise':100,'sharpness':2.0,'profile':'anime','line_detail':100,'anti_alias':100}),
    ('general-2x',{'scale':2,'denoise':20,'sharpness':1.2,'profile':'general','line_detail':0,'anti_alias':10}),
    ('real-2x',{'scale':2,'denoise':10,'sharpness':1.15,'profile':'real','line_detail':0,'anti_alias':10})
]
for case_name,params in image_cases:
    case_started=time.perf_counter()
    response=requests.post('http://127.0.0.1:8000/api/enhance/image',
        params=params,files={'file':('veloceai-self-test.png',image_bytes,'image/png')},timeout=(15,600))
    if response.status_code!=200:
        raise RuntimeError(case_name+' failed ('+str(response.status_code)+'): '+response.text[:500])
    output=Image.open(io.BytesIO(response.content)).convert('RGB')
    expected=64*int(params['scale'])
    if output.size!=(expected,expected):
        raise RuntimeError(case_name+' returned '+str(output.size)+'; expected '+str((expected,expected)))
    pixels=np.asarray(output)
    if pixels.std()<2 or int(pixels.max())-int(pixels.min())<8:
        raise RuntimeError(case_name+' returned a blank or corrupted image')
    print('[PASS] '+case_name+': '+str(output.size[0])+'x'+str(output.size[1])+'; '+str(len(response.content))+' bytes; '+format(time.perf_counter()-case_started,'.1f')+'s')
    if case_name=='anime-4x-max-controls':
        sample.save(test_dir/'sample-input.png')
        output.save(test_dir/'sample-output-4x.png')
        print('Visual quality spot-check (synthetic input / AI output):')
        display(sample.resize((256,256)),output)

video_input=test_dir/'input-test.mp4'
video_output=test_dir/'output-test.mp4'
subprocess.run([
    'ffmpeg','-y','-hide_banner','-loglevel','error',
    '-f','lavfi','-i','testsrc2=size=64x64:rate=2:duration=1',
    '-f','lavfi','-i','sine=frequency=880:sample_rate=44100:duration=1',
    '-shortest','-c:v','libx264','-pix_fmt','yuv420p','-c:a','aac',str(video_input)
],check=True)
video_params={'scale':2,'denoise':0,'sharpness':1.0,'crf':28,
              'profile':'anime','line_detail':0,'anti_alias':0,'crop_black_bars':'true'}
video_started=time.perf_counter()
with open(video_input,'rb') as video_file:
    video_response=requests.post('http://127.0.0.1:8000/api/enhance/video',
        params=video_params,files={'file':('veloceai-self-test.mp4',video_file,'video/mp4')},
        timeout=(15,600))
if video_response.status_code!=200:
    raise RuntimeError('video test failed ('+str(video_response.status_code)+'): '+video_response.text[:700])
video_output.write_bytes(video_response.content)
probe=json.loads(subprocess.check_output(['ffprobe','-v','error','-show_streams','-show_format','-of','json',str(video_output)]))
video_streams=[st for st in probe.get('streams',[]) if st.get('codec_type')=='video']
audio_streams=[st for st in probe.get('streams',[]) if st.get('codec_type')=='audio']
if not video_streams or (video_streams[0].get('width'),video_streams[0].get('height'))!=(128,128):
    raise RuntimeError('video test returned an unexpected output resolution')
if not audio_streams:
    raise RuntimeError('video test failed to preserve its audio track')
fps=video_streams[0].get('avg_frame_rate','')
if fps!='2/1':
    raise RuntimeError('video test did not preserve 2 FPS; got '+str(fps))
print('[PASS] video: 128x128; 2 FPS preserved; audio preserved; '+str(len(video_response.content))+' bytes; '+format(time.perf_counter()-video_started,'.1f')+'s')
display(Video(str(video_output),embed=True))

# End-to-end tests through the PUBLIC Cloudflare address using the same chunk/job API as the website.
def run_site_job(base_url, kind, filename, payload, settings):
    init=requests.post(base_url+'/api/upload/init',
        json={'filename':filename,'kind':kind,'size':len(payload)},timeout=(20,120))
    if init.status_code!=200:
        raise RuntimeError('Public '+kind+' upload init failed ('+str(init.status_code)+'): '+init.text[:500])
    upload_id=init.json()['upload_id']
    chunk_size=int(init.json().get('chunk_size',1024*1024))
    total=(len(payload)+chunk_size-1)//chunk_size
    for index,offset in enumerate(range(0,len(payload),chunk_size)):
        piece=payload[offset:offset+chunk_size]
        response=requests.post(base_url+'/api/upload/chunk',
            params={'upload_id':upload_id,'index':index},
            data=piece, headers={'Content-Type':'application/octet-stream'},
            timeout=(20,120))
        if response.status_code!=200:
            raise RuntimeError('Public '+kind+' upload chunk '+str(index+1)+'/'+str(total)+' failed ('+str(response.status_code)+'): '+response.text[:500])
    started=requests.post(base_url+'/api/jobs/start',
        json={'upload_id':upload_id,'settings':settings},timeout=(20,120))
    if started.status_code!=200:
        raise RuntimeError('Public '+kind+' job start failed ('+str(started.status_code)+'): '+started.text[:500])
    job_id=started.json()['job_id']
    deadline=time.time()+600
    while time.time()<deadline:
        state_response=requests.get(base_url+'/api/jobs/'+job_id,timeout=(20,60))
        if state_response.status_code!=200:
            raise RuntimeError('Public job status failed ('+str(state_response.status_code)+'): '+state_response.text[:500])
        state=state_response.json()
        if state['status']=='failed':
            raise RuntimeError('Public '+kind+' processing failed: '+str(state.get('error')))
        if state['status']=='complete':
            result=requests.get(base_url+'/api/jobs/'+job_id+'/result',timeout=(20,180))
            if result.status_code!=200 or not result.content:
                raise RuntimeError('Public '+kind+' result download failed ('+str(result.status_code)+'): '+result.text[:500])
            return result.content
        time.sleep(1)
    raise RuntimeError('Public '+kind+' test timed out while waiting for the result.')

# Prove the public tunnel can accept a file the same size as the user's 23.8 MB video.
print('Testing public tunnel transfer with a 23.8 MB multi-part upload...')
transport_bytes=os.urandom(23_800_000)
transport_init=requests.post(public_url+'/api/upload/init',
    json={'filename':'transport-check.mp4','kind':'video','size':len(transport_bytes)},
    timeout=(20,120))
if transport_init.status_code!=200:
    raise RuntimeError('23.8 MB public upload init failed ('+str(transport_init.status_code)+'): '+transport_init.text[:500])
transport_id=transport_init.json()['upload_id']
transport_chunk_size=int(transport_init.json().get('chunk_size',1024*1024))
transport_total=(len(transport_bytes)+transport_chunk_size-1)//transport_chunk_size
for transport_index,transport_offset in enumerate(range(0,len(transport_bytes),transport_chunk_size)):
    transport_part=transport_bytes[transport_offset:transport_offset+transport_chunk_size]
    transport_response=requests.post(public_url+'/api/upload/chunk',
        params={'upload_id':transport_id,'index':transport_index},
        data=transport_part, headers={'Content-Type':'application/octet-stream'},
        timeout=(20,120))
    if transport_response.status_code!=200:
        raise RuntimeError('23.8 MB tunnel upload failed at chunk '+str(transport_index+1)+'/'+str(transport_total)+': '+transport_response.text[:500])
cleanup=requests.delete(public_url+'/api/upload/'+transport_id,timeout=(20,60))
if cleanup.status_code!=200:
    raise RuntimeError('Public upload cleanup failed: '+cleanup.text[:300])
print('[PASS] public tunnel accepted 23.8 MB in '+str(transport_total)+' raw-binary chunks of '+str(transport_chunk_size//1024)+' KB each.')

print('Testing image enhancement through the public website tunnel...')
public_started=time.perf_counter()
public_image_bytes=run_site_job(public_url,'image','public-check.png',image_bytes,
    {'scale':2,'denoise':0,'sharpness':1.0,'crf':17,'profile':'anime','line_detail':20,'anti_alias':0})
public_image=Image.open(io.BytesIO(public_image_bytes)).convert('RGB')
if public_image.size!=(128,128):
    raise RuntimeError('Public-tunnel image returned '+str(public_image.size))
print('[PASS] public website image: 128x128; '+str(len(public_image_bytes))+' bytes; '+format(time.perf_counter()-public_started,'.1f')+'s')

print('Testing video enhancement through the public website tunnel...')
public_video_started=time.perf_counter()
video_bytes=video_input.read_bytes()
public_video_bytes=run_site_job(public_url,'video','public-check.mp4',video_bytes,
    {'scale':2,'denoise':0,'sharpness':1.0,'crf':28,'crop_black_bars':'false',
     'profile':'anime','line_detail':0,'anti_alias':0})
public_video_path=test_dir/'public-tunnel-output.mp4'
public_video_path.write_bytes(public_video_bytes)
public_probe=json.loads(subprocess.check_output(['ffprobe','-v','error','-show_streams','-of','json',str(public_video_path)]))
public_vstreams=[st for st in public_probe.get('streams',[]) if st.get('codec_type')=='video']
public_astreams=[st for st in public_probe.get('streams',[]) if st.get('codec_type')=='audio']
if not public_vstreams or (public_vstreams[0].get('width'),public_vstreams[0].get('height'))!=(128,128):
    raise RuntimeError('Public-tunnel video has an incorrect resolution.')
if not public_astreams or public_vstreams[0].get('avg_frame_rate')!='2/1':
    raise RuntimeError('Public-tunnel video did not preserve audio and 2 FPS.')
print('[PASS] public website video: 128x128; 2 FPS and audio preserved; '+str(len(public_video_bytes))+' bytes; '+format(time.perf_counter()-public_video_started,'.1f')+'s')
display(Video(str(public_video_path),embed=True))

print('[PASS] All VeloceAI local and public-tunnel image, video, and settings tests passed.')
print('VeloceAI website:',public_url)
try:
    print('Public tunnel health:',urllib.request.urlopen(public_url+'/health',timeout=8).read().decode())
except Exception as e:
    print('The tunnel URL was created, but its public DNS check did not complete yet:',repr(e))
    print('Open the website URL above in your browser. If it does not open, wait 30 seconds and retry; if you use a VPN, temporarily turn it off or switch network.')
print('Keep this Colab cell/session running while using the website.')
print('='*60)


## إصلاح استمرار ظهور الخطأ القديم\nحدّث دفتر Colab من الرابط نفسه، ثم شغّل خلية الإصلاح التالية وحدها (لا تستخدم Run all). ستنزّل الملفات الجديدة، تعيد تشغيل السيرفر والنفق، وتتحقق من الرابط قبل طباعته.

In [ ]:
# VeloceAI — إصلاح تحديث الواجهة وإعادة تشغيل الرابط الحالي فقط
# شغّل هذه الخلية وحدها بعد فتح/تحديث دفتر Colab. لا تستخدم Run all.

import os, sys, time, re, pathlib, subprocess, urllib.request, json

# أوقف خادمي الويب والنفق السابقين إن كانا يعملان في هذا الـ runtime.
for process_name in ("tunnel", "server"):
    process = globals().get(process_name)
    try:
        if process is not None and process.poll() is None:
            process.terminate()
            process.wait(timeout=10)
    except Exception:
        try:
            process.kill()
        except Exception:
            pass

# نزّل الملفات الحالية من GitHub مع كسر أي cache وسيط.
repo_raw = "https://raw.githubusercontent.com/rezeyuri-dot/veloceai/main/"
def download_fresh(filename, destination):
    request = urllib.request.Request(
        repo_raw + filename + "?refresh=" + str(time.time_ns()),
        headers={"Cache-Control": "no-cache", "Pragma": "no-cache"}
    )
    with urllib.request.urlopen(request, timeout=45) as response:
        data = response.read()
    pathlib.Path(destination).write_bytes(data)
    return data.decode("utf-8")

server_source = download_fresh("colab_server.py", "/content/colab_server.py")
page_source = download_fresh("index.html", "/content/index.html")

# فحوص قبل تشغيل أي شيء: لا نبدأ إن لم تكن النسخة الصحيحة قد وصلت.
required_server = [
    '@app.post("/api/upload/init")',
    '@app.post("/api/jobs/start")',
    '@app.get("/api/jobs/{job_id}")',
    'no-store, no-cache'
]
missing = [item for item in required_server if item not in server_source]
if missing:
    raise RuntimeError("نسخة السيرفر غير مكتملة؛ العناصر المفقودة: " + ", ".join(missing))
if "sendChunkedVideoRequest" in page_source:
    raise RuntimeError("ما زالت صفحة قديمة تحتوي على الدالة المفقودة.")
if "resultBlob=await sendEnhancementRequest(endpoint,data);" not in page_source:
    raise RuntimeError("إصلاح استدعاء معالجة الفيديو غير موجود في صفحة الموقع.")

# شغّل السيرفر المحدّث.
server_log = open("/content/veloceai-server.log", "w")
server = subprocess.Popen(
    [sys.executable, "/content/colab_server.py"],
    stdout=server_log, stderr=subprocess.STDOUT
)
for attempt in range(60):
    try:
        local_health = json.loads(
            urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=3).read()
        )
        if local_health.get("ok"):
            break
    except Exception:
        pass
    time.sleep(2)
else:
    print(pathlib.Path("/content/veloceai-server.log").read_text(errors="ignore")[-6000:])
    raise RuntimeError("السيرفر المحدّث لم يبدأ. راجع السجل أعلاه.")

# تأكد أن الصفحة والخدمات المطلوبة هي التي يقدمها السيرفر المحلي فعليًا.
with urllib.request.urlopen("http://127.0.0.1:8000/", timeout=10) as response:
    served_page = response.read().decode("utf-8")
    cache_header = response.headers.get("Cache-Control", "")
if "sendChunkedVideoRequest" in served_page or "resultBlob=await sendEnhancementRequest(endpoint,data);" not in served_page:
    raise RuntimeError("السيرفر ما زال يقدّم صفحة قديمة.")
if "no-store" not in cache_header:
    raise RuntimeError("تعذّر تعطيل تخزين الصفحة في المتصفح.")
openapi = json.loads(urllib.request.urlopen("http://127.0.0.1:8000/openapi.json", timeout=10).read())
needed_routes = ["/api/upload/init", "/api/upload/chunk", "/api/jobs/start",
                 "/api/jobs/{job_id}", "/api/jobs/{job_id}/result"]
missing_routes = [route for route in needed_routes if route not in openapi.get("paths", {})]
if missing_routes:
    raise RuntimeError("واجهات المعالجة غير مكتملة: " + ", ".join(missing_routes))

# حمّل النماذج داخل عملية السيرفر الجديدة قبل نشر الرابط.
for profile in ("anime", "general"):
    request = urllib.request.Request(
        "http://127.0.0.1:8000/warmup?profile=" + profile,
        data=b"", method="POST"
    )
    try:
        result = json.loads(urllib.request.urlopen(request, timeout=600).read())
        print("Model ready:", profile, result)
    except Exception as error:
        raise RuntimeError("فشل تجهيز نموذج " + profile + ": " + repr(error))

# أنشئ نفقًا جديدًا بعد إيقاف السابق.
cloudflared = "/usr/local/bin/cloudflared"
if not pathlib.Path(cloudflared).exists():
    raise RuntimeError("cloudflared غير موجود في جلسة Colab.")
tunnel_log = open("/content/veloceai-tunnel.log", "w")
tunnel = subprocess.Popen(
    [cloudflared, "tunnel", "--protocol", "http2",
     "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=tunnel_log, stderr=subprocess.STDOUT
)
public_url = None
for attempt in range(60):
    time.sleep(2)
    log_text = pathlib.Path("/content/veloceai-tunnel.log").read_text(errors="ignore")
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log_text)
    if match:
        public_url = match.group(0)
        break
if not public_url:
    print(pathlib.Path("/content/veloceai-tunnel.log").read_text(errors="ignore")[-6000:])
    raise RuntimeError("لم يتم إنشاء رابط Cloudflare جديد.")

# فحص الرابط العام: انتظر انتشار DNS أولاً، ثم اختبر الصفحة والصحة.
import socket
from urllib.parse import urlparse

host = urlparse(public_url).hostname
last_error = None
dns_ready = False
for attempt in range(36):
    try:
        socket.getaddrinfo(host, 443, type=socket.SOCK_STREAM)
        dns_ready = True
        break
    except OSError as error:
        last_error = error
        if tunnel.poll() is not None:
            break
        time.sleep(5)

if not dns_ready:
    print("تعذر حل اسم نطاق Cloudflare بعد الانتظار:", repr(last_error))
    print("حالة cloudflared:", tunnel.poll())
    print("آخر سجل للنفق:")
    print(pathlib.Path("/content/veloceai-tunnel.log").read_text(errors="ignore")[-5000:])
    print("السيرفر المحلي ما زال يعمل على http://127.0.0.1:8000")
    raise RuntimeError("فشل DNS للنفق العام؛ النماذج والسيرفر المحلي جاهزان. لا تعِد تنزيل النماذج.")

public_ok = False
for attempt in range(24):
    try:
        with urllib.request.urlopen(public_url + "/", timeout=20) as response:
            public_page = response.read().decode("utf-8")
            public_cache = response.headers.get("Cache-Control", "")
        health = json.loads(urllib.request.urlopen(public_url + "/health", timeout=20).read())
        assert "sendChunkedVideoRequest" not in public_page
        assert "resultBlob=await sendEnhancementRequest(endpoint,data);" in public_page
        assert "no-store" in public_cache
        assert health.get("ok")
        public_ok = True
        break
    except Exception as error:
        last_error = error
        if tunnel.poll() is not None:
            break
        time.sleep(5)

if not public_ok:
    print("آخر خطأ في فحص الرابط:", repr(last_error))
    print("آخر سجل للنفق:")
    print(pathlib.Path("/content/veloceai-tunnel.log").read_text(errors="ignore")[-5000:])
    raise RuntimeError("النفق لم يصبح متاحاً بعد. السيرفر المحلي والنماذج جاهزة؛ أرسل سجل النفق أعلاه.")


# End-to-end public export verification: 23.8 MB raw upload + processed MP4 + downloaded result.
import requests, tempfile

def _upload_raw_chunks(base_url, filename, kind, payload):
    init = requests.post(base_url + "/api/upload/init",
        json={"filename": filename, "kind": kind, "size": len(payload)}, timeout=(20, 120))
    if init.status_code != 200:
        raise RuntimeError("Upload init failed: " + str(init.status_code) + " " + init.text[:400])
    upload_id = init.json()["upload_id"]
    chunk_size = int(init.json()["chunk_size"])
    try:
        total = (len(payload) + chunk_size - 1) // chunk_size
        for index, offset in enumerate(range(0, len(payload), chunk_size)):
            part = payload[offset:offset + chunk_size]
            response = requests.post(base_url + "/api/upload/chunk",
                params={"upload_id": upload_id, "index": index},
                data=part, headers={"Content-Type": "application/octet-stream"},
                timeout=(20, 120))
            if response.status_code != 200:
                raise RuntimeError("Chunk " + str(index + 1) + "/" + str(total) +
                    " failed: " + str(response.status_code) + " " + response.text[:400])
        return upload_id, total
    except Exception:
        requests.delete(base_url + "/api/upload/" + upload_id, timeout=(10, 30))
        raise

# Verify transfer size matching the previously failing 23.8 MB phone video.
large_test_payload = os.urandom(23_800_000)
large_upload_id, large_chunk_count = _upload_raw_chunks(
    public_url, "transport-check.mp4", "video", large_test_payload)
cleanup_response = requests.delete(public_url + "/api/upload/" + large_upload_id, timeout=(10, 30))
if cleanup_response.status_code != 200:
    raise RuntimeError("Large-upload cleanup failed: " + cleanup_response.text[:300])
print("[PASS] Public tunnel accepted 23.8 MB using " + str(large_chunk_count) +
      " raw-binary chunks (" + str(transport_chunk_size // 1024) + " KB each).")

# Build a tiny MP4 with audio, process it through the same public chunk/job/result routes as the website.
export_dir = pathlib.Path("/content/veloceai-export-selftest")
export_dir.mkdir(exist_ok=True)
test_input = export_dir / "input-with-audio.mp4"
test_output = export_dir / "VeloceAI-Restored.mp4"
subprocess.run([
    "ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
    "-f", "lavfi", "-i", "testsrc2=size=64x64:rate=2:duration=1",
    "-f", "lavfi", "-i", "sine=frequency=880:sample_rate=44100:duration=1",
    "-shortest", "-c:v", "libx264", "-pix_fmt", "yuv420p",
    "-c:a", "aac", str(test_input)
], check=True)
test_video_bytes = test_input.read_bytes()
export_upload_id, export_chunk_count = _upload_raw_chunks(
    public_url, "export-check.mp4", "video", test_video_bytes)
start_response = requests.post(public_url + "/api/jobs/start",
    json={"upload_id": export_upload_id, "settings": {
        "scale": 2, "denoise": 0, "sharpness": 1.0, "crf": 28,
        "profile": "anime", "line_detail": 0, "anti_alias": 0,
        "crop_black_bars": "false"
    }}, timeout=(20, 120))
if start_response.status_code != 200:
    raise RuntimeError("Video export job could not start: " +
                       str(start_response.status_code) + " " + start_response.text[:500])
export_job_id = start_response.json()["job_id"]
export_deadline = time.time() + 900
while time.time() < export_deadline:
    job_response = requests.get(public_url + "/api/jobs/" + export_job_id, timeout=(20, 60))
    if job_response.status_code != 200:
        raise RuntimeError("Video export status failed: " +
                           str(job_response.status_code) + " " + job_response.text[:400])
    job_state = job_response.json()
    if job_state.get("status") == "failed":
        raise RuntimeError("Video export failed: " + str(job_state.get("error")))
    if job_state.get("status") == "complete":
        break
    time.sleep(2)
else:
    raise RuntimeError("Video export self-test timed out after 15 minutes.")
result_response = requests.get(
    public_url + "/api/jobs/" + export_job_id + "/result", timeout=(20, 180))
if result_response.status_code != 200 or not result_response.content:
    raise RuntimeError("Final video result could not be downloaded: " +
                       str(result_response.status_code) + " " + result_response.text[:400])
test_output.write_bytes(result_response.content)
probe = json.loads(subprocess.check_output([
    "ffprobe", "-v", "error", "-show_streams", "-show_format", "-of", "json", str(test_output)
]))
video_streams = [stream for stream in probe.get("streams", []) if stream.get("codec_type") == "video"]
audio_streams = [stream for stream in probe.get("streams", []) if stream.get("codec_type") == "audio"]
if not video_streams:
    raise RuntimeError("Downloaded export has no video stream.")
if (video_streams[0].get("width"), video_streams[0].get("height")) != (128, 128):
    raise RuntimeError("Downloaded export resolution is incorrect.")
if video_streams[0].get("avg_frame_rate") != "2/1":
    raise RuntimeError("Downloaded export did not preserve 2 FPS.")
if not audio_streams:
    raise RuntimeError("Downloaded export did not preserve the audio track.")
print("[PASS] Public video export completed; result downloaded (" +
      str(len(result_response.content)) + " bytes), 128x128, 2 FPS, audio preserved.")
print("[PASS] Exported MP4 saved at:", str(test_output))

print("\n" + "="*55)
print("[PASS] الصفحة العامة هي النسخة المحدّثة، وليست النسخة المخزّنة القديمة.")
print("[PASS] واجهات رفع الأجزاء والمهام الخلفية موجودة.")
print("[PASS] Cache-Control:", public_cache)
print("رابط VeloceAI الجديد:", public_url)
print("Health:", health)
print("اترك خلية Colab هذه والجلسة تعمل أثناء استخدام الموقع.")
print("="*55)
